In [4]:
# import the nedded libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
import pickle

In [5]:
# create Customer Base
num_customer= 200
customer_id= np.arange(1, num_customer + 1)

In [6]:
# create customer behavior features
total_orders = np.random.randint(1, 20, num_customer)
total_spent = total_orders * np.random.uniform(20, 500, num_customer)

In [7]:
## Feature Engineering and Categorical Data

# avg order value + country
avg_order_value= total_spent / total_orders
countries= ['Germany', 'France', 'uk']
country= np.random.choice(countries, num_customer) 

In [9]:
# target variable creations
high_value= (total_spent > 3000).astype(int)

In [10]:
# build data frame
df= pd.DataFrame ({
    'customer_id': customer_id,
    'total_orders': total_orders,
    'total_spent': total_spent,
    'avg_order_value': avg_order_value,
    'country': country,
    'high_value': high_value
})

In [11]:
## Exploratory Data Analysis (EDA)

In [12]:
df.head()

,customer_id,total_orders,total_spent,avg_order_value,country,high_value
0,1,12,4791.753958,399.312830,Germany,1
1,2,5,802.744718,160.548944,France,0
2,3,5,985.009835,197.001967,uk,0
3,4,3,783.703091,261.234364,Germany,0
4,5,13,4441.008648,341.616050,Germany,1


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      200 non-null    int64  
 1   total_orders     200 non-null    int32  
 2   total_spent      200 non-null    float64
 3   avg_order_value  200 non-null    float64
 4   country          200 non-null    object 
 5   high_value       200 non-null    int64  
dtypes: float64(2), int32(1), int64(2), object(1)
memory usage: 8.7+ KB


In [14]:
df.describe()

,customer_id,total_orders,total_spent,avg_order_value,high_value
count,200.000000,200.000000,200.000000,200.000000,200.000000
mean,100.500000,10.105000,2739.487078,263.381722,0.400000
std,57.879185,5.513253,2157.522521,128.695409,0.491127
min,1.000000,1.000000,50.147060,21.247880,0.000000
25%,50.750000,6.000000,823.393959,164.802546,0.000000
50%,100.500000,10.000000,2323.664622,262.482633,0.000000
75%,150.250000,15.000000,4031.055396,367.968888,1.000000
max,200.000000,19.000000,8741.556072,498.749412,1.000000


In [15]:
## Label Encoding
le = LabelEncoder()
df['country_encoded']= le.fit_transform(df['country'])

In [16]:
df[['country', 'country_encoded']].head()

,country,country_encoded
0,Germany,1
1,France,0
2,uk,2
3,Germany,1
4,Germany,1


In [17]:
## Train-Test Split
X = df[['total_orders', 'total_spent', 'avg_order_value', 'country_encoded']]
y = df[['high_value']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state= 42
)

In [18]:
## Decision Tree Model
model= DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [19]:
print(model)

DecisionTreeClassifier(random_state=42)


In [20]:
## Model Evaluation (Accuracy)
y_pred = model.predict(X_test)
accuracy= accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 1.0


In [21]:
## Classification preformance Evaluaton with Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[21  0]
 [ 0 19]]


In [22]:
## KNN Model

# data scaling
scaler= StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [23]:
# K-Nearest Neighbors Model
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn= knn.predict(X_test_scaled)

C:\Users\yaras\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\neighbors\_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


In [24]:
# accuracy for KNN
accuracy_knn= accuracy_score(y_test, y_pred_knn)
print("KNN Accuracy:", accuracy_knn)

KNN Accuracy: 0.875


In [25]:
# confusion matrix (KNN)
cm_knn= confusion_matrix(y_test, y_pred_knn)
print(cm_knn)

[[17  4]
 [ 1 18]]


## Model Comparison and Insights

Two models were compared: Decision Tree and K-Nearest Neighbors.

The Decision Tree model achieved higher accuracy (77.5%) compared to KNN (67.5%) and performed better in identifying high-value customers.

The confusion matrix analysis showed that KNN missed more high-value customers, making it less suitable for this business scenario.

From a business perspective, correctly identifying high-value customers is critical, making the Decision Tree the preferred model.

In [26]:
# save model
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

# save scaler
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# save encoder:
with open("encoder.pkl", "wb") as f:
    pickle.dump(le, f)

In [31]:
# Load everything
model = pickle.load(open("model.pkl", "rb"))
scaler = pickle.load(open("scaler.pkl", "rb"))
encoder = pickle.load(open("encoder.pkl", "rb"))

def predict_customer(total_orders, total_spent, avg_order_value, country):
    
    # Encode country
    country_encoded = encoder.transform([country])[0]
    
    # Create input
    X = np.array([[total_orders, total_spent, avg_order_value, country_encoded]])
    
    # Predict
    prediction = model.predict(X)[0]
    
    return "High Value" if prediction == 1 else "Low Value"

In [ ]:
# Example
predict_customer(10, 5000, 500, "Germany")

C:\Users\yaras\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


'High Value'